# Chapter 3: Machine Learning Techniques

Companion notebook for Chapter 3 of *Mastering AI-Powered Finance* (`Overleaf/Book 1`). It was built from Chapter 3 of the full manuscript (Machine Learning Techniques); the source references below point to that manuscript.

Companion to `Overleaf/Final Project/Chapter 3 - Machine Learning Techniques/main.tex`.

This notebook turns the chapter code examples into a student-facing workflow. Code cells follow the order used in the chapter and retain a reproducible offline demonstration alongside the optional World Bank download.

Run from top to bottom with this chapter folder as the working directory. The default mode generates **synthetic data**, with GDP per capita deliberately constructed as a noisy linear function of life expectancy, urbanization, and enrollment. These are invented economies, not World Bank observations; results do not establish which model works best on real countries. The revised manuscript does not assert an empirical score table. The existing bundled `data/gdp_per_capita_full.csv` is also synthetic.

Set `USE_LIVE_WORLD_BANK = True` for the 2020 World Bank cross-section. This needs internet access and `wbgapi`; it excludes regional/income aggregates and drops economies missing a required indicator. Complete-case selection can bias the sample, and live revisions can change both its size and model scores. Synthetic and downloaded data use separate filenames.


In [1]:
# Common notebook setup
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
# Grayscale defaults, matching the printed book: black, mid and light gray,
# each with its own line style so series stay distinguishable in print.
from cycler import cycler
plt.rcParams['axes.prop_cycle'] = cycler(color=['0.0', '0.45', '0.7'], linestyle=['-', '--', ':'])
plt.rcParams['image.cmap'] = 'Greys'


## AI Tutorial / Predicting GDP per Capita from Socio-Economic Indicators

### Code example 1
Source: `Overleaf/Final Project/Chapter 3 - Machine Learning Techniques/main.tex`


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

Path('data').mkdir(exist_ok=True)
USE_LIVE_WORLD_BANK = False  # set True to download current World Bank data with wbgapi

columns = [
    'Primary enrollment', 'Urban population %', 'Life expectancy',
    'Inflation %', 'Unemployment %', 'Exports % GDP', 'Imports % GDP',
    'FDI inflows % GDP', 'GDP growth %', 'Household consumption %',
    'Tax revenue % GDP'
]

if USE_LIVE_WORLD_BANK:
    import wbgapi as wb
    inds = [
      'NY.GDP.PCAP.KD', 'SE.PRM.ENRR', 'SP.URB.TOTL.IN.ZS',
      'SP.DYN.LE00.IN', 'FP.CPI.TOTL.ZG', 'SL.UEM.TOTL.ZS',
      'NE.EXP.GNFS.ZS', 'NE.IMP.GNFS.ZS', 'BX.KLT.DINV.WD.GD.ZS',
      'NY.GDP.MKTP.KD.ZG', 'NE.CON.PRVT.ZS', 'GC.TAX.TOTL.GD.ZS'
    ]
    df = wb.data.DataFrame(
        inds, time=2020, labels=True, skipAggs=True, db=2
    ).reset_index()
    indicator_mapping_short = {
      'NY.GDP.PCAP.KD': 'GDP per capita',
      'SE.PRM.ENRR': 'Primary enrollment',
      'SP.URB.TOTL.IN.ZS': 'Urban population %',
      'SP.DYN.LE00.IN': 'Life expectancy',
      'FP.CPI.TOTL.ZG': 'Inflation %',
      'SL.UEM.TOTL.ZS': 'Unemployment %',
      'NE.EXP.GNFS.ZS': 'Exports % GDP',
      'NE.IMP.GNFS.ZS': 'Imports % GDP',
      'BX.KLT.DINV.WD.GD.ZS': 'FDI inflows % GDP',
      'NY.GDP.MKTP.KD.ZG': 'GDP growth %',
      'NE.CON.PRVT.ZS': 'Household consumption %',
      'GC.TAX.TOTL.GD.ZS': 'Tax revenue % GDP'
    }
    df = df.rename(columns=indicator_mapping_short)
    available_economies = len(df)
    df = df.dropna(subset=columns + ['GDP per capita']).copy()
    data_path = Path('data/gdp_per_capita_world_bank_2020.csv')
    data_source = 'World Bank WDI, 2020 (latest available revisions)'
    print(f'Complete cases: {len(df)} of {available_economies} economies')
else:
    data_path = Path('data/gdp_per_capita_synthetic.csv')
    data_source = 'Synthetic demonstration (not World Bank observations)'
    rng = np.random.default_rng(42)
    n = 190
    df = pd.DataFrame({
        'economy': [f'Country {i:03d}' for i in range(n)],
        'Primary enrollment': rng.normal(95, 12, n).clip(45, 130),
        'Urban population %': rng.uniform(20, 98, n),
        'Life expectancy': rng.normal(72, 8, n).clip(48, 86),
        'Inflation %': rng.normal(4, 5, n).clip(-3, 35),
        'Unemployment %': rng.normal(7, 4, n).clip(1, 30),
        'Exports % GDP': rng.uniform(10, 95, n),
        'Imports % GDP': rng.uniform(12, 100, n),
        'FDI inflows % GDP': rng.normal(3, 4, n).clip(-5, 25),
        'GDP growth %': rng.normal(2.5, 3, n).clip(-12, 12),
        'Household consumption %': rng.uniform(35, 85, n),
        'Tax revenue % GDP': rng.uniform(8, 38, n),
    })
    df['GDP per capita'] = (
        450 * df['Life expectancy'] + 180 * df['Urban population %']
        + 120 * df['Primary enrollment'] + rng.normal(0, 5000, n)
    ).clip(700, 80000)

df.to_csv(data_path, index=False)
print(data_source)
print(f'Saved {df.shape[0]} rows to {data_path}')
df.head()

Synthetic demonstration (not World Bank observations)
Saved 190 rows to data/gdp_per_capita_synthetic.csv


,economy,Primary enrollment,Urban population %,Life expectancy,Inflation %,Unemployment %,Exports % GDP,Imports % GDP,FDI inflows % GDP,GDP growth %,Household consumption %,Tax revenue % GDP,GDP per capita
0,Country 000,98.656605,84.826788,78.755082,-0.443987,3.427478,20.111031,84.435204,4.615816,3.490870,72.303399,9.378897,65200.870887
1,Country 001,82.520191,31.956177,72.091408,3.646598,7.045805,87.932115,26.784075,5.261842,1.864799,79.285259,18.566375,47115.647003
2,Country 002,104.005414,33.982928,82.631685,5.671476,5.802941,37.330304,24.924009,10.344903,3.996158,45.796240,29.342280,54071.070440
3,Country 003,106.286777,66.751858,78.854352,4.255710,2.939728,61.690983,87.750486,2.188232,-3.821577,61.564638,8.553134,53219.635094
4,Country 004,71.587578,88.215839,78.734561,0.172324,15.195023,49.526201,84.434966,4.504177,2.370599,73.963497,21.175355,50011.167311


## AI Tutorial / Machine Learning Modeling

### Code example 2
Source: `Overleaf/Final Project/Chapter 3 - Machine Learning Techniques/main.tex`

The random split evaluates predictions for held-out economies in the same year; it is **not a future GDP forecast or a causal analysis**. GDP-normalized predictors share information with the target's underlying accounts, and contemporaneous indicators may only be released or revised later. A forecasting exercise needs lagged, availability-dated features and a chronological evaluation across multiple years.

SVM and KNN depend on feature scales, so their scalers are fitted inside training pipelines. The SVM also standardizes its training target: otherwise its default regularization and epsilon are poorly matched to GDP measured in dollars. Predictions are automatically converted back to the target's original units. A training-mean baseline makes the error scores easier to judge. These fixed model configurations illustrate a workflow; a single test split does not establish a universal model ranking. Tune on training-only cross-validation and reserve a separate final test set for model selection work.


In [3]:
# Machine learning part
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
import pandas as pd

# Load from CSV
df = pd.read_csv(data_path)
print(f"Data source: {data_source}")

X = df[[
  "Primary enrollment", "Urban population %",
  "Life expectancy", "Inflation %", "Unemployment %",
  "Exports % GDP", "Imports % GDP", "FDI inflows % GDP",
  "GDP growth %", "Household consumption %", "Tax revenue % GDP"
]]
y = df["GDP per capita"]

print(f"Dataset shape: {df.shape}")
print(f"Number of economies: {df['economy'].nunique()}")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

models = {
  "Training-mean baseline": DummyRegressor(strategy="mean"),
  "SVM": TransformedTargetRegressor(
      regressor=make_pipeline(StandardScaler(), SVR()),
      transformer=StandardScaler()
  ),
  "Linear Regression": LinearRegression(),
  "Decision Tree": DecisionTreeRegressor(random_state=0),
  "KNN": make_pipeline(StandardScaler(), KNeighborsRegressor()),
  "Random Forest": RandomForestRegressor(random_state=0),
  "XGBoost": XGBRegressor(random_state=0)
}

for name, model in models.items():
  model.fit(X_tr, y_tr)
  pred = model.predict(X_te)
  mae = mean_absolute_error(y_te, pred)
  rmse = root_mean_squared_error(y_te, pred)
  r2 = model.score(X_te, y_te)
  print(f"{name} - MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.3f}")

Data source: Synthetic demonstration (not World Bank observations)
Dataset shape: (190, 13)
Number of economies: 190
Training-mean baseline - MAE: 5409.20, RMSE: 6974.54, R2: -0.073
SVM - MAE: 4096.42, RMSE: 5265.96, R2: 0.389
Linear Regression - MAE: 4037.41, RMSE: 5193.28, R2: 0.405
Decision Tree - MAE: 6088.69, RMSE: 7373.35, R2: -0.199
KNN - MAE: 4352.76, RMSE: 5820.11, R2: 0.253
Random Forest - MAE: 4107.97, RMSE: 5366.64, R2: 0.365


XGBoost - MAE: 4371.69, RMSE: 5854.75, R2: 0.244
